Testing The Script

In [2]:
class UnionFind:
    def __init__(self, elements):
        """Initialize Union-Find with path compression."""
        self.parent = {x: x for x in elements}  # Each element is its own parent
        self.rank = {x: 1 for x in elements}    # Rank for union by rank

    def find(self, x):
        """Find with path compression."""
        if self.parent[x] != x:
            self.parent[x] = self.find(self.parent[x])  # Path compression
        return self.parent[x]

    def union(self, x, y):
        """Union by rank."""
        root_x = self.find(x)
        root_y = self.find(y)
        
        if root_x != root_y:
            if self.rank[root_x] > self.rank[root_y]:
                self.parent[root_y] = root_x
            elif self.rank[root_x] < self.rank[root_y]:
                self.parent[root_x] = root_y
            else:
                self.parent[root_y] = root_x
                self.rank[root_x] += 1

    def __repr__(self):
        return f"Parent: {self.parent}\nRank: {self.rank}"

In [91]:
import numpy as np
from itertools import combinations
from tqdm import tqdm

# Translated originally form Julia Code, we generate adinkras via cubical cohomology

def generate_code(generator):
    d = len(generator)
    N = len(generator[0])
    
    # collect all combinations of length 1..d
    combs = []
    for dd in range(1, d+1):
        combs.extend(combinations(generator, dd))
    
    code = []
    for str_tuple in combs:
        # XOR all vectors in the tuple
        acc = str_tuple[0].copy()
        for vec in str_tuple[1:]:
            acc = np.bitwise_xor(acc, vec)
        code.append(acc)
    
    # prepend the zero vector
    code.insert(0, np.zeros(N, dtype=np.uint8))
    return code

def sigmaprod(arr_1d):
    s0 = np.array([[1,0],[0,1]])
    s1 = np.array([[0,1],[1,0]])
    if arr_1d.shape[0] > 2:
        prod = sigmaprod(arr_1d[1:])
        return np.kron([s0,s1][arr_1d[0]], prod)
    else:
        return np.kron([s0,s1][arr_1d[0]],[s0,s1][arr_1d[1]])

def GRDN(Gammas):
    KronDelta = np.eye(len(Gammas), dtype=np.int8)
    Identity = np.eye(Gammas[0].shape[0], dtype=np.int8)
    for I_, gamma_I in enumerate(Gammas):
        for J_, gamma_J in enumerate(Gammas):
            assert np.matmul(gamma_I,gamma_J).shape == (KronDelta[I_,J_] * Identity).shape
            assert np.array_equal(np.matmul(gamma_I,gamma_J) + np.matmul(gamma_J,gamma_I), 2*KronDelta[I_,J_] * Identity)



def decode_ncube_signed(generator, allow_large_computations=False):
    d = len(generator)
    N = len(generator[0])
    
    if N >= 16 and not allow_large_computations:
        return "Are You Sure?"
    
    code = generate_code(generator)
    
    # NCubeLattice = all binary vectors of length N
    NCubeLattice = [np.array(list(map(int, np.binary_repr(nn, width=N))), dtype=np.bool)
                    for nn in range(2**N)]

    # Make an array of indices
    indices = list(range(len(NCubeLattice)))
    # Sort the indices based on the Nth entry of each word
    indices = sorted(indices, key=lambda i: NCubeLattice[i][-1])
    indices_lookup = [0 for i in indices]
    for idx,i in enumerate(indices):
        indices_lookup[i] = idx

    NCubeLattice = [NCubeLattice[i] for i in indices]

    NCubeLatticeBool = np.array([1 for _ in range(2**N)])
    # Basis = standard basis vectors
    Basis = [np.array(list(map(int, np.binary_repr(1 << nn, width=N))), dtype=np.bool)
             for nn in range(N)]

    NCubeDashing = []
    
    # Dashings are applied to every lattice point and their basis vectors to make things straightforward
    # Naturally both fermions and bosons will have correct dashedness by the definition given in 'Codes and Supersymmetry in One Dimension'
    for lattice_point in tqdm(NCubeLattice):
        solid_v_dashed = [(-1)**np.count_nonzero(np.atleast_1d(lattice_point[:N-nn-1])) for nn in range(N)]
        NCubeDashing.append(solid_v_dashed)
    NCubeDashing_arr = np.array(NCubeDashing)

    #Sign_vectors = [NCubeDashing_arr.T[basis] for basis in range(len(Basis))]
    #Permutations = [sigmaprod(basis.astype(np.int8))[indices,:][:,indices] for basis in Basis]
    #Sign_Matrices = [np.zeros((2**N,2**N)) for _ in range(N)]

    #for n,vec in enumerate(Sign_vectors):
    #    for i,val in enumerate(vec):
    #        Sign_Matrices[n][i,i] = val

    #A_Ncube = [np.matmul(p,s) for p,s in zip(Permutations, Sign_Matrices)]
    #GRDN(A_Ncube)

    NCubeAdjlist = np.array([[[i, indices_lookup[int(np.dot(np.bitwise_xor(point, basis), 2**np.arange(N-1, -1, -1)))], NCubeDashing_arr[i,nn]] for i,point in enumerate(NCubeLattice)] for nn,basis in tqdm(enumerate(Basis), total=N)])

    #good
    Partitioned_dict = {}
    Partitions = np.zeros((2**(N-d), len(code)), dtype = np.int32)
    Partitioned_inv_dict = {}
    # Partition NCube

    for new_point in tqdm(range(2**(N-d))):

        # apply XOR with each codeword
        next_existing_lattice_point = np.where(NCubeLatticeBool == 1)[0][0]
        new_partition = [np.bitwise_xor(NCubeLattice[next_existing_lattice_point], c).astype(np.bool) for c in code]
        assert np.array_equal(new_partition[0], NCubeLattice[next_existing_lattice_point].astype(np.bool))
        Partitioned_dict[new_point] = new_partition
        # reindex the divided lattice as a new set of points, log the relational data for instant lookup
        idxs = np.zeros(len(new_partition), dtype = np.int32)
        for hi,point in enumerate(new_partition):
            Partitioned_inv_dict[point.tobytes()] = new_point
            idx = indices_lookup[np.dot(point, 2**np.arange(N-1, -1, -1))]
            idxs[hi] = idx
            # delete corresponding lattice point
            #print(idx)
            NCubeLatticeBool[idx] = 0
        Partitions[new_point] = idxs

    Partitions = Partitions.T
    pb = Partitions[0]
    sign_group = []
    for i,pi in tqdm(enumerate(Partitions[1:]), total=len(code)-1):
        sign_group.append([])
        for nn in range(N):
            assert abs(int(np.dot(NCubeAdjlist[nn,pb,2], NCubeAdjlist[nn,pi,2]))) == len(pb) # all edges must be correct up to the same sign
            if int(np.dot(NCubeAdjlist[nn,pb,2], NCubeAdjlist[nn,pi,2])) == -len(pb): # NOT detects a sign disagreement.
                sign_group[i].append(nn)

    print("Starting UnionFind")
    nodes_for_disjoint_union = np.concatenate(Partitions)
    uf = UnionFind(nodes_for_disjoint_union)
    for i,pi in tqdm(enumerate(Partitions[1:]), total=len(code)-1):
        connections_therein = NCubeAdjlist[np.array(sign_group[i])][:,pi][:,:,:2] # edge connections between all nodes outside
        for nn in range(len(sign_group[i])): 
            for n_a, n_b in connections_therein[nn]:
                uf.union(n_a, n_b)
    roots = {x: uf.find(x) for x in nodes_for_disjoint_union} 
    roots_ = {}
    for key, val in roots.items():
        if val not in roots_:
            roots_[val] = set()
        roots_[val].add(key)
    print(roots_)


    # Adjacency structure
    Adjacency = [[np.zeros(2**(N-d), dtype=np.int8) for _ in range(2**(N-d))]
                  for _ in range(N)]
    for nn, basis_vec in tqdm(enumerate(Basis),total = len(Basis)):
        for new_point in range(2**(N-d)):
            newstr = np.bitwise_xor(Partitioned_dict[new_point][0], basis_vec).astype(np.bool).tobytes()
            if newstr in Partitioned_inv_dict:
                Adjacency[nn][new_point][Partitioned_inv_dict[newstr]] = NCubeAdjlist[nn, pb[new_point], 2]
    AdjacencyC = np.array(Adjacency)
    AdjacencyM = np.sum(AdjacencyC, axis=0)
    try:
        GRDN(AdjacencyC)
    except:
        print("Fails GRDN Algebra")

    return(AdjacencyC,AdjacencyM)

In [92]:
codegen = [[1, 1, 1, 1]]
#codegen = [[0,0,1,1,1,1,0,0], [1,1,1,1,0,0,0,0], [0,0,0,0,1,1,1,1], [1,0,1,0,1,0,1,0]]
#codegen = [[1,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0], [0,0,1,1,1,1,0,0,0,0,0,0,0,0,0,0], [0,0,0,0,1,1,1,1,0,0,0,0,0,0,0,0], [0,0,0,0,0,0,1,1,1,1,0,0,0,0,0,0], [0,0,0,0,0,0,0,0,1,1,1,1,0,0,0,0], [0,0,0,0,0,0,0,0,0,0,1,1,1,1,0,0], [0,0,0,0,0,0,0,0,0,0,0,0,1,1,1,1], [1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0]]

In [93]:
adjc, adj = decode_ncube_signed(codegen, allow_large_computations=True)
#print(adj)

100%|██████████| 1/1 [00:00<00:00, 3880.02it/s]


Starting UnionFind


100%|██████████| 1/1 [00:00<00:00, 9218.25it/s]


{np.int32(10): {np.int32(0), np.int32(8), np.int32(2), np.int32(10)}, np.int32(11): {np.int32(11), np.int32(1), np.int32(3), np.int32(9)}, np.int32(14): {np.int32(4), np.int32(12), np.int32(6), np.int32(14)}, np.int32(15): {np.int32(13), np.int32(15), np.int32(5), np.int32(7)}}


100%|██████████| 4/4 [00:00<00:00, 33222.21it/s]

Fails GRDN Algebra


In [68]:
complex(np.sqrt(2)/2,np.sqrt(2)/2) * complex(np.sqrt(2)/2,np.sqrt(2)/2)

1.0000000000000002j

In [49]:
print(np.matmul(adjc[2],adjc[3]) + np.matmul(adjc[3],adjc[2]))


[[0 0 0 2 0 0 0 0]
 [0 0 2 0 0 0 0 0]
 [0 2 0 0 0 0 0 0]
 [2 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 2]
 [0 0 0 0 0 0 2 0]
 [0 0 0 0 0 2 0 0]
 [0 0 0 0 2 0 0 0]]


In [254]:
import numpy as np
import networkx as nx

def get_distance_matrix(adjacencyC):
    """
    Compute the distance matrix of a graph defined by adjacency matrices (AdjacencyC).
    
    Parameters
    ----------
    adjacencyC : list of np.ndarray
        List of adjacency matrices (square, binary).
    
    Returns
    -------
    dm : np.ndarray
        Distance matrix (num_vertices x num_vertices).
    """
    n = adjacencyC[0].shape[0]
    
    # --- Build edge set ---
    edges = set()
    for mt in adjacencyC:
        # neighbors for each row
        oneline = mt @ np.arange(1, n+1)   # same as Julia's mt*[1..n]
        for i, val in enumerate(oneline):
            if val > 0:   # has neighbor(s)
                # Find all j such that edge exists
                neighbors = np.where(mt[i] > 0)[0]
                for j in neighbors:
                    edge = tuple(sorted((i, j)))
                    if edge[0] != edge[1]:
                        edges.add(edge)
    
    # --- Build graph ---
    G = nx.Graph()
    G.add_nodes_from(range(n))
    G.add_edges_from(edges)
    
    # --- Compute all-pairs shortest paths ---
    dist = dict(nx.all_pairs_shortest_path_length(G))
    
    # Convert to numpy matrix
    dm = np.full((n, n), np.inf)
    for i in range(n):
        dm[i, i] = 0
        for j, d in dist[i].items():
            dm[i, j] = d
    
    return dm.astype(int)


In [255]:
def even_odd_weight_sort(lst):
    evens = [x for x in lst if x % 2 == 0]
    odds  = [x for x in lst if x % 2 == 1]
    return evens + odds

In [92]:
def custom_sort(lst):
    binlist = [bin(x) for x in lst]
    evens = []
    odds = []
    for x,b in zip(lst,binlist):
        if b.count("1") % 2 == 0:
            evens.append(x)
        else:
            odds.append(x)
    return evens + odds


In [114]:
import os
import re
def ConvertToLRMatrices(Gammas):
    lrBlockMats = []
    for gamma in Gammas:
        d = gamma.shape[0]
        assert d//2 == d/2
        reindex = np.array(custom_sort(list(range(gamma.shape[0]))))
        new_ = gamma[reindex].T
        new__ = new_[reindex].T
        lrBlockMats.append(new__)
    Ls = [lrBlockMats[i][:d//2,d//2:] for i in range(len(Gammas))]
    Rs = [lrBlockMats[i][d//2:,:d//2] for i in range(len(Gammas))]
    return [Ls,Rs]

def write_Adinkra_format(filepath, Ls, Rs):
    line = ['"',"{","{"]
    for i,L in enumerate(Ls):
        line.append("{")
        for j,row in enumerate(L.astype(np.int8)):
            s = str(row)
            s = s.strip("[]")                  # "  1  0 1"
            s = re.sub(r"\s+", ",", s)         # ",1,0,1"
            s = s.strip(",")                   # "1,0,1"
            s = "{" + s + "}"
            line.append(s)
            if j != len(L)-1:
                line.append(",")
        line.append("}")
        if i != len(Ls)-1:
            line.append(",")
    line.append("}")
    line.append(",")
    line.append("{")
    for i,R in enumerate(Rs):
        line.append("{")
        for j,row in enumerate(R.astype(np.int8)):
            s = str(row)
            s = s.strip("[]")                  # "  1  0 1"
            s = re.sub(r"\s+", ",", s)         # ",1,0,1"
            s = s.strip(",")                   # "1,0,1"
            s = "{" + s + "}"
            line.append(s)
            if j != len(R)-1:
                line.append(",")
        line.append("}")
        if i != len(Rs)-1:
            line.append(",")
    line += ["}","}",'"']
 
    with open(os.path.join(filepath), "w") as f:
        f.write("".join(line))
        f.close()

In [225]:
Ls, Rs = ConvertToLRMatrices(adjc_ncube)
write_Adinkra_format("NCube.csv", Ls, Rs)

In [ ]:

L_s, R_s = ConvertToLRMatrices(adjc)
write_Adinkra_format("PNCube.csv", L_s, R_s)
L_s, R_s = ConvertToLRMatrices(adjc_ncube2)
write_Adinkra_format("NCube2.csv", L_s, R_s)

In [124]:
def GRDN(Gammas):
    KronDelta = np.eye(len(Gammas))
    Identity = np.eye(Gammas[0].shape[0])
    for I_, gamma_I in enumerate(Gammas):
        for J_, gamma_J in enumerate(Gammas):
            assert np.matmul(gamma_I,gamma_J).shape == (KronDelta[I_,J_] * Identity).shape
            assert np.array_equal(np.matmul(gamma_I,gamma_J) + np.matmul(gamma_J,gamma_I), 2*KronDelta[I_,J_] * Identity)


In [125]:
def ComplexGRDN(Gammas):
    KronDelta = np.eye(len(Gammas))
    Identity = np.eye(Gammas[0].shape[0])
    try:
        for I_, gamma_I in enumerate(Gammas):
            for J_, gamma_J in enumerate(Gammas):
                assert np.matmul(gamma_I,gamma_J).shape == (KronDelta[I_,J_] * Identity).shape
                assert np.allclose((np.matmul(gamma_I, gamma_J) + np.matmul(gamma_J, gamma_I)), 2*KronDelta[I_,J_] * Identity)
    except:

        print((np.matmul(gamma_I, gamma_J) + np.conjugate(np.matmul(gamma_J, gamma_I))).real)

In [226]:
GRDN(adjc_Ncube)

In [15]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

def distance_matrix_plot(distM):
    """
    Plot a distance matrix as a black/white heatmap (white=low, black=high).
    
    Parameters
    ----------
    distM : np.ndarray
        Distance matrix.
    """
    miv = np.min(distM)
    mav = np.max(distM)
    
    # Colormap: white → black, midpoint at (miv + mav)/2
    midpoint = miv + (mav - miv) / 2
    cmap = LinearSegmentedColormap.from_list("bw_map", ["white", "black"])
    
    plt.figure(figsize=(4, 4))
    im = plt.imshow(distM, cmap=cmap, vmin=miv, vmax=mav, origin='upper')
    plt.axis("off")   # hide axes
    plt.box(True)     # frame box, like Julia's framestyle=:box
    plt.show()


In [ ]:

def string_to_nested_list(string_representation):
    """
    Converts a string representation of a nested list to an actual nested list.

    Args:
        string_representation: The string to convert.

    Returns:
        A nested list.
    """
    try:
        return ast.literal_eval(string_representation)
    except (SyntaxError, ValueError):
        return "Invalid string format for nested list"


# Adinkra class
class Adinkra:
    def __init__(self):
            self.boson_positions = None
            self.fermion_positions = None
            self.boson_labels = None
            self.fermion_labels = None
            self.edge_colors = None
            self.boson_elevations = None
            self.fermion_elevations = None
            self.adinkra_colors = None
            self.adinkra_size = None
            self.edges = None
            self.dashing = None

    def open(self, path : str):
        if os.path.exists(os.path.join(path)):
            self.path = path
            # For Table (assuming space-separated)
            with open(self.path, "r") as file:
                text = file.readlines()
            for line in text:
                line= line.replace("}", "]").replace("{", "[").replace("\n", "").replace("\"", "").replace("\'", "")
                list = string_to_nested_list(line)
                Ls = np.array(list[0])
                print(Ls[3])
                num_colors = Ls.shape[0]
                num_bosons = Ls.shape[1]
                num_fermions = Ls.shape[2]
                self.boson_elevations = np.ones(num_bosons)
                self.fermion_elevations = np.zeros(num_fermions)
                # Create a list of colored edges of the form [*colors: [*connections [*boson, *fermion, *dashing: +/-1]]...]
                edges = np.array([[[j, np.nonzero(Ls[i,j])[0][0], Ls[i,j,np.nonzero(Ls[i,j])[0][0]]] for j in range(num_bosons)] for i in range(num_colors)])
                self.adinkra_colors = num_colors
                self.adinkra_size = (num_bosons, num_fermions)
                self.edges = edges[:, :, 0:2]
                self.dashing = edges[:, :, 2]
        else:
            print(f"File {path} does not exist.")

    def GRDN(self, Gammas):
        KronDelta = np.eye(len(Gammas))
        Identity = np.eye(Gammas[0].shape[0])
        for I_, gamma_I in enumerate(Gammas):
            for J_, gamma_J in enumerate(Gammas):
                assert np.matmul(gamma_I,gamma_J).shape == (KronDelta[I_,J_] * Identity).shape
                assert np.array_equal(np.matmul(gamma_I,gamma_J) + np.matmul(gamma_J,gamma_I), 2*KronDelta[I_,J_] * Identity)

    def decode_ncube_signed(self, generator, allow_large_computations=False):
        d = len(generator)
        N = len(generator[0])
        
        if N >= 16 and not allow_large_computations:
            return "Are You Sure?"
        
        code = generate_code(generator)
        
        # NCubeLattice = all binary vectors of length N
        NCubeLattice = [np.array(list(map(int, np.binary_repr(nn, width=N))), dtype=np.bool)
                        for nn in range(2**N)]

        # Make an array of indices
        indices = list(range(len(NCubeLattice)))
        # Sort the indices based on the Nth entry of each word
        indices = sorted(indices, key=lambda i: NCubeLattice[i][-1])
        indices_lookup = [0 for i in indices]
        for idx,i in enumerate(indices):
            indices_lookup[i] = idx

        NCubeLattice = [NCubeLattice[i] for i in indices]

        NCubeLatticeBool = np.array([1 for _ in range(2**N)])
        # Basis = standard basis vectors
        Basis = [np.array(list(map(int, np.binary_repr(1 << nn, width=N))), dtype=np.bool)
                for nn in range(N)]

        NCubeDashing = []
        
        # Dashings are applied to every lattice point and their basis vectors to make things straightforward
        # Naturally both fermions and bosons will have correct dashedness by the definition given in 'Codes and Supersymmetry in One Dimension'
        for lattice_point in tqdm(NCubeLattice):
            solid_v_dashed = [(-1)**np.count_nonzero(np.atleast_1d(lattice_point[:N-nn-1])) for nn in range(N)]
            NCubeDashing.append(solid_v_dashed)
        NCubeDashing_arr = np.array(NCubeDashing)

        Sign_vectors = [NCubeDashing_arr.T[basis] for basis in range(len(Basis))]
        Permutations = [sigmaprod(basis.astype(np.int8))[indices,:][:,indices] for basis in Basis]
        Sign_Matrices = [np.zeros((2**N,2**N)) for _ in range(N)]

        for n,vec in enumerate(Sign_vectors):
            for i,val in enumerate(vec):
                Sign_Matrices[n][i,i] = val

        A_Ncube = [np.matmul(p,s) for p,s in zip(Permutations, Sign_Matrices)]
        self.GRDN(A_Ncube)

        NCubeAdjlist = np.array([[[i, indices_lookup[int(np.dot(np.bitwise_xor(point, basis), 2**np.arange(N-1, -1, -1)))], NCubeDashing_arr[i,nn]] for i,point in enumerate(NCubeLattice)] for nn,basis in tqdm(enumerate(Basis), total=N)])

        NCubeAdjMatbef = np.zeros((N,2**N,2**N))
        for nn in range(N):
            for row in NCubeAdjlist[nn]:
                NCubeAdjMatbef[nn,row[0], row[1]] = row[2]
        
        GRDN(NCubeAdjMatbef)
        
        #good
        Partitioned_dict = {}
        Partitions = np.zeros((2**(N-d), len(code)), dtype = np.int32)
        Partitioned_inv_dict = {}
        # Partition NCube

        for new_point in tqdm(range(2**(N-d))):

            # apply XOR with each codeword
            next_existing_lattice_point = np.where(NCubeLatticeBool == 1)[0][0]
            new_partition = [np.bitwise_xor(NCubeLattice[next_existing_lattice_point], c).astype(np.bool) for c in code]
            assert np.array_equal(new_partition[0], NCubeLattice[next_existing_lattice_point].astype(np.bool))
            Partitioned_dict[new_point] = new_partition
            # reindex the divided lattice as a new set of points, log the relational data for instant lookup
            idxs = np.zeros(len(new_partition), dtype = np.int32)
            for hi,point in enumerate(new_partition):
                Partitioned_inv_dict[point.tobytes()] = new_point
                idx = indices_lookup[np.dot(point, 2**np.arange(N-1, -1, -1))]
                idxs[hi] = idx
                # delete corresponding lattice point
                print(idx)
                NCubeLatticeBool[idx] = 0
            Partitions[new_point] = idxs

        Partitions = Partitions.T
        pb = Partitions[0]
        for i,pi in tqdm(enumerate(Partitions[1:]), total=len(code)-1):
            for nn in range(N):
                assert abs(int(np.dot(NCubeAdjlist[nn,pb,2], NCubeAdjlist[nn,pi,2]))) == len(pb) # all edges must be correct up to the same sign
                if int(np.dot(NCubeAdjlist[nn,pb,2], NCubeAdjlist[nn,pi,2])) == -len(pb): # detects a sign disagreement.
                    mask_a = np.isin(NCubeAdjlist[nn, :, 0], pi)
                    mask_b = np.isin(NCubeAdjlist[nn, :, 0], pb) & np.isin(NCubeAdjlist[nn, :, 1], pi)
                    ### Here is where my code fails: the quotient operation and sign correction is meant to make all self-identifying edges become the same sign.
                    NCubeAdjlist[nn, mask_a, 2] = -1*NCubeAdjlist[nn, mask_a, 2]
                    NCubeAdjlist[nn, mask_b, 2] = -1*NCubeAdjlist[nn, mask_b, 2]
                    if np.any(mask_b):
                        print(mask_b)
                        print(nn,i)
                    
        NCubeAdjMataft = np.zeros((N,2**N,2**N))
        for nn in range(N):
            for row in NCubeAdjlist[nn]:
                NCubeAdjMataft[nn,row[0], row[1]] = row[2]
        
        # Adjacency structure
        Adjacency = [[np.zeros(2**(N-d), dtype=int) for _ in range(2**(N-d))]
                    for _ in range(N)]
        for nn, basis_vec in tqdm(enumerate(Basis),total = len(Basis)):
            for new_point in range(2**(N-d)):
                newstr = np.bitwise_xor(Partitioned_dict[new_point][0], basis_vec).astype(np.bool).tobytes()
                if newstr in Partitioned_inv_dict:
                    Adjacency[nn][new_point][Partitioned_inv_dict[newstr]] = NCubeAdjlist[nn, pb[new_point], 2]
        AdjacencyC = np.array(Adjacency)
        AdjacencyM = np.sum(AdjacencyC, axis=0)
        try:
            self.GRDN(AdjacencyC)
        except:
            print("Fails GRDN Algebra")

        return(AdjacencyC,AdjacencyM, A_Ncube, NCubeAdjMataft)
    
    def create_from_cube(N,k):
        ## Heres where I would put my award if I had one
        return
    def __repr__(self):
        return f"Adinkra ({self.adinkra_size[0]}x{self.adinkra_size[1]}) at path: {self.path}\nBoson elevations: {self.boson_elevations}\nFermion elevations: {self.fermion_elevations}" 


